# BMW Sales Classification Project (2010-2024)

In [1]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from kaggle.api.kaggle_api_extended import KaggleApi
from scipy.stats import chi2_contingency, zscore
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix, classification_report
)


## Data Loading and Overviewing

In [7]:
def load_kaggle_dataset(dataset, path):
    api = KaggleApi()
    api.authenticate()
    
    os.makedirs(path, exist_ok=True)

    api.dataset_download_files(dataset, path=path, unzip=True)

    files = os.listdir(path)
    csv_files = [f for f in files if f.lower().endswith(".csv")]

    if not csv_files:
        raise FileNotFoundError("CSV files not found in the dataset!")

    csv_path = os.path.join(path, csv_files[0])
    print(f"Loading file→ {csv_files[0]}")
   
    return pd.read_csv(csv_path)

path_to_data = "../data/bmw_sales"

df_bmw_sales = load_kaggle_dataset(
    "ahmadrazakashif/bmw-worldwide-sales-records-20102024",
    path=path_to_data
)

Dataset URL: https://www.kaggle.com/datasets/ahmadrazakashif/bmw-worldwide-sales-records-20102024
Loading file→ BMW sales data (2010-2024) (1).csv


In [8]:
df_bmw_sales = pd.read_csv("bmw-worldwide-sales-records-20102024/BMW sales data (2010-2024) (1).csv")

In [9]:
df_bmw_sales.head(10)

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low
5,5 Series,2017,Middle East,Silver,Diesel,Manual,1.9,171362,42926,1232,Low
6,i8,2022,Europe,White,Diesel,Manual,1.8,196741,55064,7949,High
7,M5,2014,Asia,Black,Diesel,Automatic,1.6,121156,102778,632,Low
8,X3,2016,South America,White,Diesel,Automatic,1.7,48073,116482,8944,High
9,i8,2019,Europe,White,Electric,Manual,3.0,35700,96257,4411,Low


In [ ]:
df_bmw_sales.shape

In [ ]:
df_bmw_sales.info()

In [ ]:
df_bmw_sales.describe()

In [ ]:
df_bmw_sales.nunique()

In [ ]:
df_bmw_sales.isna().sum()

In [ ]:
df_bmw_sales.duplicated().sum()

## EDA (Exploratory Data Analysis)

In [ ]:
target = 'Sales_Classification' 
numeric_cols = df_bmw_sales.select_dtypes(exclude='object').columns 
cat_cols = [col for col in df_bmw_sales.select_dtypes(include='object').columns if col != target]

In [ ]:
sns.countplot(x=target, data=df_bmw_sales)
plt.title(f'Class Distribution: {target}')
plt.show()

**Observation:** 

The target is imbalanced, so we will use `class_weight='balanced'` in Random Forest  to ensure the model correctly accounts for both classes.



In [ ]:
for col in cat_cols:
    print(f"{col}:\n{df_bmw_sales[col].value_counts(normalize=True)}\n")

In [ ]:
for col in cat_cols:
    cross = pd.crosstab(df_bmw_sales[col], df_bmw_sales['Sales_Classification'], normalize='index')
    cross.plot(kind='bar', stacked=True)
    plt.title(f'{col} vs Sales_Classification')
    plt.show()

In [ ]:
for col in cat_cols:
    contingency = pd.crosstab(df_bmw_sales[col], df_bmw_sales[target])
    chi2, p, dof, ex = chi2_contingency(contingency)
    print(f"{col}: p-value = {p}")

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(4,3))
    sns.boxplot(x=target, y=col, data=df_bmw_sales)
    plt.title(f'{col} vs {target}')
    plt.show()


In [ ]:
df_bmw_sales.groupby('Sales_Classification')['Sales_Volume'].agg(['min', 'max'])

### Observation

**Sales_Volume ranges by class:**
- **Low:** 100–6,999  
- **High:** 7,000–9,999  

The split between *Low* and *High* aligns almost perfectly with this boundary.  
This means that `Sales_Volume` almost completely separates the target classes —  
a strong indication of **data leakage** or **a** **target-derived feature**.  

For now, the feature will remain in the dataset to confirm this hypothesis  
after model evaluation.


In [ ]:
for col in numeric_cols:
    sns.histplot(df_bmw_sales[col], kde=True)
    plt.title(f'{col} Distribution')
    plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df_bmw_sales[numeric_cols].corr(), annot=True, cmap='YlGnBu')
plt.title('Correlation Heatmap')
plt.show() 

In [ ]:
for col in numeric_cols:
    z_scores = zscore(df_bmw_sales[col].dropna())
    outliers = sum(abs(z_scores) > 3)
    print(f'{col}: {outliers} potential outliers')


## Data Cleaning & Preprocessing

In [ ]:
X = df_bmw_sales.drop(columns=['Sales_Classification'])
y = df_bmw_sales['Sales_Classification'].map({'Low': 0, 'High': 1})

In [ ]:
# `Year` transformed into `Years_Since_First_Sale` to represent a time trend
df_bmw_sales['Years_Since_First_Sale'] = df_bmw_sales['Year'] - df_bmw_sales['Year'].min()

In [ ]:
numeric_features = ['Engine_Size_L', 'Mileage_KM', 'Price_USD', 'Sales_Volume','Years_Since_First_Sale']
categorical_features = ['Model', 'Region', 'Color', 'Fuel_Type', 'Transmission']

In [ ]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

## Modeling

In [ ]:
# Baseline Model
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

In [ ]:
baseline_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_baseline = baseline_pipeline.predict(X_test)

In [ ]:
# GridSearch Model
param_grid = {
        'clf__n_estimators':[100,200],
        'clf__max_depth':[None,10,20],
        'clf__min_samples_split':[2,5],

        'clf__min_samples_leaf':[1,2],
    }

In [ ]:
grid_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

In [ ]:
grid_search = GridSearchCV(
    estimator=grid_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',     
    cv=5,               
    n_jobs=-1,         
    verbose=2          
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
y_pred_grid = grid_search.predict(X_test)

## Model Evaluation

In [ ]:
param_keys = ['max_depth', 'min_samples_leaf', 'min_samples_split', 'n_estimators']
baseline_params = {
    key: baseline_pipeline.named_steps['clf'].get_params()[key] for key in param_keys
}

In [ ]:
print("Best parameters Basic Model:", baseline_params)
print("Best parameters Grid Model:", grid_search.best_params_)

In [ ]:
baseline_metrics = {
    'Model': 'Baseline',
    'Accuracy': accuracy_score(y_test, y_pred_baseline),
    'ROC AUC': roc_auc_score(y_test, y_pred_baseline),
    'F1': f1_score(y_test, y_pred_baseline),
    'Precision': precision_score(y_test, y_pred_baseline),
    'Recall': recall_score(y_test, y_pred_baseline)
}

In [ ]:
grid_metrics = {
    'Model': 'GridSearch',
    'Accuracy': accuracy_score(y_test, y_pred_grid),
    'ROC AUC': roc_auc_score(y_test, y_pred_grid),
    'F1': f1_score(y_test, y_pred_grid),
    'Precision': precision_score(y_test, y_pred_grid),
    'Recall': recall_score(y_test, y_pred_grid)
}

In [ ]:
metrics_df = pd.DataFrame([baseline_metrics, grid_metrics])
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.heatmap(confusion_matrix(y_test, y_pred_baseline), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Baseline Confusion Matrix')
sns.heatmap(confusion_matrix(y_test, y_pred_grid), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('GridSearch Confusion Matrix')
plt.show()

**Observation:**

Both baseline and tuned Random Forest models achieved perfect metrics (Accuracy, ROC AUC, F1 = 1.0).  
Such performance is highly unusual and suggests possible **data leakage** or the presence of a dominant feature  
that directly determines the target variable.  

To confirm this, we will review the results of the GridSearchCV parameter combinations  
to check whether all configurations yield the same performance.


In [ ]:
print("Total combinations tested:", len(grid_search.cv_results_['params']))
print("ROC AUC for each combination:")
for mean, params in zip(grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['params']):
    print(f"{mean:.4f} -> {params}")



## Conclusions / Interpretation

In [ ]:
baseline_pipeline = grid_search.best_estimator_

In [ ]:
feature_names = baseline_pipeline.named_steps['preprocessor'].get_feature_names_out()

In [ ]:
importances = baseline_pipeline.named_steps['clf'].feature_importances_

In [ ]:
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feat_imp.head(10))
plt.title('Top 10 Feature Importances (GridSearchCV Model)')
plt.show()

**Observation:**

Feature importance analysis shows that `Sales_Volume` overwhelmingly dominates the model,  
accounting for nearly all predictive power. This confirms earlier EDA findings,  
where `Sales_Volume` almost completely separated the target classes.  

Such dominance strongly suggests **data leakage**, as the target variable (`Sales_Classification`)  
was likely derived from this feature.  

**Next step:**  
    
Rebuild the model **without `Sales_Volume`** to verify whether the model can still generalize  
and learn meaningful patterns from the remaining features.



## Model Refinement (Without Sales_Volume)

In [ ]:
X_refined = X.drop(columns=['Sales_Volume'])

In [ ]:
X_train_ref, X_test_ref, y_train_ref, y_test_ref = train_test_split(
    X_refined, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
numeric_cols_refined = [col for col in numeric_cols if col != 'Sales_Volume']

In [ ]:
preprocessor_refined = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols_refined),
        ('cat', categorical_transformer, cat_cols)
    ]
)

In [ ]:
# Refined Basic Model 
baseline_pipeline_refined = Pipeline([
    ('preprocessor', preprocessor_refined),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

In [ ]:
baseline_pipeline_refined.fit(X_train_ref, y_train_ref)

In [ ]:
y_pred_baseline_ref = baseline_pipeline_refined.predict(X_test_ref)

In [ ]:
# Refined Grid Model 
grid_pipeline_refined = Pipeline([
    ('preprocessor', preprocessor_refined),
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])


In [ ]:
grid_search_refined = GridSearchCV(
    estimator=grid_pipeline_refined,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [ ]:
grid_search_refined.fit(X_train_ref, y_train_ref)

In [ ]:

y_pred_grid_ref = grid_search_refined.predict(X_test_ref)

## Model Evaluation (Without Sales_Volume)


In [ ]:
baseline_params_ref = {
    key: baseline_pipeline_refined.named_steps['clf'].get_params()[key] for key in param_keys
}

In [ ]:
print("Best parameters Refined Basic Model:", baseline_params_ref)
print("Best parameters Refined Grid Model:", grid_search_refined.best_params_)

In [ ]:
baseline_metrics_ref = {
    'Model': 'Refined Baseline ',
    'Accuracy': accuracy_score(y_test, y_pred_baseline_ref),
    'ROC AUC': roc_auc_score(y_test, y_pred_baseline_ref),
    'F1': f1_score(y_test, y_pred_baseline_ref),
    'Precision': precision_score(y_test, y_pred_baseline_ref),
    'Recall': recall_score(y_test, y_pred_baseline_ref)
}

In [ ]:
grid_metrics_ref = {
    'Model': 'Refined GridSearch',
    'Accuracy': accuracy_score(y_test, y_pred_grid_ref),
    'ROC AUC': roc_auc_score(y_test, y_pred_grid_ref),
    'F1': f1_score(y_test, y_pred_grid_ref),
    'Precision': precision_score(y_test, y_pred_grid_ref),
    'Recall': recall_score(y_test, y_pred_grid_ref)
}

In [ ]:
metrics_df_refined = pd.DataFrame([
    baseline_metrics,
    grid_metrics,
    baseline_metrics_ref,
    grid_metrics_ref
])
metrics_df_refined

**Observation:**

After removing `Sales_Volume`, the model performance dropped dramatically.
This confirms that `Sales_Volume` was overwhelmingly driving predictions and likely introduced **data leakage**.  
The model without this feature shows that the remaining variables alone are not sufficient for accurate classification,  
highlighting the importance of careful feature selection and validation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
sns.heatmap(confusion_matrix(y_test, y_pred_baseline_ref), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Refined Baseline Confusion Matrix')
sns.heatmap(confusion_matrix(y_test, y_pred_grid_ref), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Refined GridSearch Confusion Matrix')
plt.show()

## Conclusions

1. **Data Overview & EDA**  
   - Target classes were imbalanced.  
   - `Sales_Volume` almost perfectly separates Low and High, suggesting potential data leakage.


2. **Modeling with Sales_Volume**  
   - Baseline and GridSearch models achieved perfect metrics.  
   - Feature importance confirmed that `Sales_Volume` dominated predictions.


3. **Modeling without Sales_Volume**  
   - Performance dropped drastically: Accuracy ~0.57–0.69, ROC AUC ~0.5.  
   - Confusion matrices show poor recognition of the positive class.  
   - Feature importance is more evenly distributed, but remaining features are insufficient for accurate predictions.
    

4. **Key Takeaways**  
   - Importance of careful EDA and checking for data leakage.  
   - Dominant features can mask the weakness of other predictors.  
   - Always validate models by testing feature influence and robustness.
